# Field Mapping 04 - Spatial Correlation Analysis

**Report Date:** March 2026  
**Dataset:** Oregon Willamette Valley Agricultural Fields (50 fields)


## Section 1: Setup & Configuration

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import rasterio
from pyproj import Transformer
from scipy import stats
import json
import os
import random

# Set random seed for reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Configuration Parameters
MIN_PIXEL_COUNT = 30
SATELLITE_METRICS = ['ndvi', 'msavi', 'evi', 'ndmi']
SOIL_PROPERTIES = ['ph', 'om_pct', 'clay_pct', 'sand_pct', 'cec', 'awc', 'silt_pct']
TERRAIN_PROPERTIES = ['elevation', 'slope', 'aspect']

# Define paths
PROJECT_ROOT = '/workspaces/ag-skills-demo'
FIELDS_FILE = os.path.join(PROJECT_ROOT, 'data/fields_oregon_willamette_ag_2025.geojson')
SATELLITE_DIR = os.path.join(PROJECT_ROOT, 'data/satellite')
TERRAIN_DIR = os.path.join(PROJECT_ROOT, 'data/terrain')
SOIL_DIR = os.path.join(PROJECT_ROOT, 'data/soil')
SOIL_PROPS_DIR = os.path.join(PROJECT_ROOT, 'data/soil_properties')
TERRAIN_RESAMPLED_DIR = os.path.join(PROJECT_ROOT, 'data/terrain_resampled')
SOIL_PROPS_RESAMPLED_DIR = os.path.join(PROJECT_ROOT, 'data/soil_properties_resampled')

# Display name mappings
SATELLITE_DISPLAY = {'ndvi': 'NDVI', 'msavi': 'MSAVI', 'evi': 'EVI', 'ndmi': 'NDMI'}
TERRAIN_DISPLAY = {'elevation': 'Elevation (m)', 'slope': 'Slope (deg)', 'aspect': 'Aspect (deg)'}
SOIL_PROPERTY_DISPLAY = {
    'ph': 'pH', 
    'om_pct': 'Organic Matter (%)', 
    'clay_pct': 'Clay (%)', 
    'sand_pct': 'Sand (%)', 
    'cec': 'CEC (meq/100g)',
    'awc': 'AWC (cm/cm)',
    'silt_pct': 'Silt (%)'
}

SAT_CMAPS = {'ndvi': 'RdYlGn', 'msavi': 'viridis', 'evi': 'viridis', 'ndmi': 'RdBu'}
TERRAIN_CMAPS = {'elevation': 'terrain', 'slope': 'YlOrRd', 'aspect': 'hsv'}
SOIL_CMAPS = {'ph': 'coolwarm', 'om_pct': 'YlGn', 'clay_pct': 'YlOrBr', 'sand_pct': 'YlGnBu', 'cec': 'Purples', 'awc': 'Blues', 'silt_pct': 'YlOrRd'}

print('Configuration loaded:')
print(f'  MIN_PIXEL_COUNT: {MIN_PIXEL_COUNT}')
print(f'  SATELLITE_METRICS: {SATELLITE_METRICS}')
print(f'  SOIL_PROPERTIES: {SOIL_PROPERTIES}')
print(f'  TERRAIN_PROPERTIES: {TERRAIN_PROPERTIES}')


Configuration loaded:
  MIN_PIXEL_COUNT: 30
  SATELLITE_METRICS: ['ndvi', 'msavi', 'evi', 'ndmi']
  SOIL_PROPERTIES: ['ph', 'om_pct', 'clay_pct', 'sand_pct', 'cec', 'awc', 'silt_pct']
  TERRAIN_PROPERTIES: ['elevation', 'slope', 'aspect']


## Section 2: Field Selection

In [2]:
# Load field boundaries
gdf = gpd.read_file(FIELDS_FILE)
print(f'Loaded {len(gdf)} fields from {FIELDS_FILE}')
print(f'CRS: {gdf.crs}')

# Select 4 random fields
selected_fields = random.sample(list(gdf['field_id']), 4)
print(f'\nSelected fields (seed={RANDOM_SEED}): {selected_fields}')

# Display field info
field_info = gdf[gdf['field_id'].isin(selected_fields)][['field_id', 'area_acres', 'lat', 'lon']]
print('\nField Information:')
print(field_info.to_string(index=False))


Loaded 50 fields from /workspaces/ag-skills-demo/data/fields_oregon_willamette_ag_2025.geojson
CRS: EPSG:5070

Selected fields (seed=42): ['WV_AG_041', 'WV_AG_008', 'WV_AG_002', 'WV_AG_048']

Field Information:
 field_id  area_acres       lat         lon
WV_AG_002   66.794644 44.188688 -123.183989
WV_AG_008   13.584710 44.592012 -122.698694
WV_AG_041   10.785175 44.450616 -123.102804
WV_AG_048   15.112003 44.184532 -123.171003


## Section 3: Helper Functions

In [3]:
def load_tiff_with_nodata(tiff_path):
    """Load TIFF file and return data with nan substituted for nodata."""
    if not os.path.exists(tiff_path):
        return None, None
    with rasterio.open(tiff_path) as src:
        data = src.read(1)
        nodata = src.nodata if src.nodata is not None else -9999
        plot_data = np.where(data == nodata, np.nan, data)
    return plot_data, src

def get_field_boundary_pixels(field_id, ref_raster_path):
    """Get field boundary in pixel coordinates for a given raster."""
    global gdf
    
    field_geom = gdf[gdf['field_id'] == field_id].geometry.iloc[0]
    
    if not os.path.exists(ref_raster_path):
        return None
    
    with rasterio.open(ref_raster_path) as src:
        transform = src.transform
        raster_crs = src.crs
    
    field_crs = gdf.crs
    transformer = Transformer.from_crs(field_crs, raster_crs, always_xy=True)
    
    if field_geom.geom_type == 'Polygon':
        coords = [list(field_geom.exterior.coords)]
    elif field_geom.geom_type == 'MultiPolygon':
        coords = [list(p.exterior.coords) for p in field_geom.geoms]
    else:
        return None
    
    pixel_coords = []
    for poly in coords:
        poly_pixels = []
        for lon, lat in poly:
            x, y = transformer.transform(lon, lat)
            col, row = ~transform * (x, y)
            poly_pixels.append((col, row))
        pixel_coords.append(poly_pixels)
    
    return pixel_coords

def calculate_pixel_correlation(array1, array2, min_pixels=MIN_PIXEL_COUNT):
    """Calculate Pearson correlation between two arrays, handling NoData."""
    valid_mask = ~np.isnan(array1) & ~np.isnan(array2)
    valid_count = np.sum(valid_mask)
    if valid_count < min_pixels:
        return np.nan, np.nan, valid_count
    x = array1[valid_mask]
    y = array2[valid_mask]
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan, np.nan, valid_count
    r, p = stats.pearsonr(x, y)
    return r, p, valid_count

def get_10m_data(field_id, data_type, property_name=None):
    """Get 10m resolution data for correlation analysis.
    
    Args:
        field_id: Field identifier
        data_type: 'satellite', 'terrain', or 'soil'
        property_name: Specific property name
    """
    if data_type == 'satellite':
        path = f'{SATELLITE_DIR}/{field_id}_{property_name}.tif'
    elif data_type == 'terrain':
        # Use original 5m terrain data
        path = f'{TERRAIN_DIR}/{field_id}_{property_name}.tif'
    elif data_type == 'soil':
        path = f'{SOIL_PROPS_DIR}/{field_id}_soil_{property_name}.tif'
    else:
        return None
    
    return load_tiff_with_nodata(path)

print('Helper functions loaded.')


Helper functions loaded.


## Section 4: Image Display

Display satellite, terrain, and soil property images with field boundaries overlaid. 
Use the dropdown to select which field to display.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

def plot_field_images(field_id):
    """Plot all images for a single field with field boundaries."""
    # Get reference raster paths for boundary extraction
    sat_ref = f'{SATELLITE_DIR}/{field_id}_ndvi.tif'
    terr_ref = f'{TERRAIN_DIR}/{field_id}_elevation.tif'
    soil_ref = f'{SOIL_PROPS_DIR}/{field_id}_soil_ph.tif'
    
    # Get field boundaries in pixel coordinates for each raster
    bounds_sat = get_field_boundary_pixels(field_id, sat_ref)
    bounds_terr = get_field_boundary_pixels(field_id, terr_ref)
    bounds_soil = get_field_boundary_pixels(field_id, soil_ref)
    
    # --- Satellite Images (2x2) ---
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    fig.suptitle(f'{field_id} - Satellite Metrics', fontsize=14, fontweight='bold')
    
    for idx, m in enumerate(SATELLITE_METRICS):
        ax = axes[idx//2, idx%2]
        png_path = f'{SATELLITE_DIR}/{field_id}_{m}.png'
        
        if os.path.exists(png_path):
            img = plt.imread(png_path)
            ax.imshow(img)
        else:
            data, src = load_tiff_with_nodata(f'{SATELLITE_DIR}/{field_id}_{m}.tif')
            if data is not None:
                vmin = np.nanpercentile(data, 2)
                vmax = np.nanpercentile(data, 98)
                ax.imshow(data, cmap=SAT_CMAPS.get(m, 'viridis'), vmin=vmin, vmax=vmax)
        
        # Add field boundary overlay
        if bounds_sat:
            for poly in bounds_sat:
                xs = [c[0] for c in poly]
                ys = [c[1] for c in poly]
                ax.plot(xs, ys, 'r-', lw=2)
        
        ax.set_title(SATELLITE_DISPLAY.get(m, m.upper()), fontsize=12, fontweight='bold')
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # --- Terrain Images (1x3) ---
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(f'{field_id} - Terrain Properties', fontsize=14, fontweight='bold')
    
    for idx, p in enumerate(TERRAIN_PROPERTIES):
        ax = axes[idx]
        png_path = f'{TERRAIN_DIR}/{field_id}_{p}.png'
        
        if os.path.exists(png_path):
            img = plt.imread(png_path)
            ax.imshow(img)
        else:
            data, src = load_tiff_with_nodata(f'{TERRAIN_DIR}/{field_id}_{p}.tif')
            if data is not None:
                if p == 'aspect':
                    vmin, vmax = 0, 360
                else:
                    vmin = np.nanpercentile(data, 2)
                    vmax = np.nanpercentile(data, 98)
                ax.imshow(data, cmap=TERRAIN_CMAPS.get(p, 'viridis'), vmin=vmin, vmax=vmax)
        
        # Add field boundary overlay
        if bounds_terr:
            for poly in bounds_terr:
                xs = [c[0] for c in poly]
                ys = [c[1] for c in poly]
                ax.plot(xs, ys, 'r-', lw=2)
        
        ax.set_title(TERRAIN_DISPLAY.get(p, p.capitalize()), fontsize=12, fontweight='bold')
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # --- Soil Property Images (3x3 grid) ---
    n_props = len(SOIL_PROPERTIES)
    n_rows = (n_props + 2) // 3
    fig, axes = plt.subplots(n_rows, 3, figsize=(15, 5 * n_rows))
    fig.suptitle(f'{field_id} - Soil Properties', fontsize=14, fontweight='bold')
    
    for idx, p in enumerate(SOIL_PROPERTIES):
        row = idx // 3
        col = idx % 3
        ax = axes[row, col] if n_rows > 1 else axes[col]
        
        data, src = load_tiff_with_nodata(f'{SOIL_PROPS_DIR}/{field_id}_soil_{p}.tif')
        
        if data is not None:
            valid = data[~np.isnan(data)]
            if len(valid) > 0:
                vmin = np.nanpercentile(data, 2)
                vmax = np.nanpercentile(data, 98)
            else:
                vmin, vmax = None, None
            
            im = ax.imshow(data, cmap=SOIL_CMAPS.get(p, 'viridis'), vmin=vmin, vmax=vmax)
            plt.colorbar(im, ax=ax, shrink=0.8)
        
        # Add field boundary overlay
        if bounds_soil:
            for poly in bounds_soil:
                xs = [c[0] for c in poly]
                ys = [c[1] for c in poly]
                ax.plot(xs, ys, 'r-', lw=2)
        
        ax.set_title(SOIL_PROPERTY_DISPLAY.get(p, p), fontsize=11, fontweight='bold')
        ax.axis('off')
    
    # Hide empty subplots
    for idx in range(n_props, n_rows * 3):
        row = idx // 3
        col = idx % 3
        if n_rows > 1:
            axes[row, col].axis('off')
        else:
            axes[col].axis('off')
    
    plt.tight_layout()
    plt.show()

# Create interactive dropdown
field_dropdown = widgets.Dropdown(
    options=selected_fields,
    description='Field:',
    style={'description_width': 'initial'},
    value=selected_fields[0]
)

output = widgets.Output()
_init_mode_img = True  # Prevent callback during init

def on_field_change(change):
    with output:
        clear_output(wait=True)
        plot_field_images(change['new'])

field_dropdown.observe(on_field_change, names='value')

print('Select a field from the dropdown to view images.')
display(widgets.VBox([
    widgets.HTML('<h3>Field Image Viewer</h3>'),
    field_dropdown,
    output
]))

print("Select a field from the dropdown to view images.")

Select a field from the dropdown to view images.


Select a field from the dropdown to view images.


## Section 5: Histograms

Histograms showing the distribution of pixel values within each field for satellite metrics and terrain properties.
Note: Soil properties are not included as they are too coarse (soil data is at 5m resolution with fewer unique values per field).

In [5]:
def plot_field_histograms(field_id):
    """Plot histograms for satellite and terrain properties of a field."""
    
    # --- Satellite Metrics Histograms ---
    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    fig.suptitle(f'{field_id} - Satellite Metrics Distribution', fontsize=14, fontweight='bold')
    
    for idx, metric in enumerate(SATELLITE_METRICS):
        ax = axes[idx // 2, idx % 2]
        data, src = load_tiff_with_nodata(f'{SATELLITE_DIR}/{field_id}_{metric}.tif')
        
        if data is not None:
            valid_data = data[~np.isnan(data)]
            ax.hist(valid_data, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
            mean_val = np.mean(valid_data)
            std_val = np.std(valid_data)
            ax.axvline(mean_val, color='red', linestyle='--', lw=2, label=f'mean={mean_val:.3f}')
            ax.text(0.02, 0.98, f'n={len(valid_data)}\nσ={std_val:.3f}', 
                    transform=ax.transAxes, fontsize=9, verticalalignment='top',
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        ax.set_title(SATELLITE_DISPLAY.get(metric, metric.upper()), fontsize=12, fontweight='bold')
        ax.set_xlabel('Value')
        ax.set_ylabel('Frequency')
        ax.legend(loc='upper right', fontsize=8)
    
    plt.tight_layout()
    plt.show()
    
    # --- Terrain Histograms ---
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f'{field_id} - Terrain Properties Distribution', fontsize=14, fontweight='bold')
    
    for idx, prop in enumerate(TERRAIN_PROPERTIES):
        ax = axes[idx]
        data, src = load_tiff_with_nodata(f'{TERRAIN_DIR}/{field_id}_{prop}.tif')
        
        if data is not None:
            valid_data = data[~np.isnan(data)]
            ax.hist(valid_data, bins=30, edgecolor='black', alpha=0.7, color='forestgreen')
            mean_val = np.mean(valid_data)
            std_val = np.std(valid_data)
            ax.axvline(mean_val, color='red', linestyle='--', lw=2, label=f'mean={mean_val:.2f}')
            ax.text(0.02, 0.98, f'n={len(valid_data)}\nσ={std_val:.2f}', 
                    transform=ax.transAxes, fontsize=9, verticalalignment='top',
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        ax.set_title(TERRAIN_DISPLAY.get(prop, prop.capitalize()), fontsize=12, fontweight='bold')
        ax.set_xlabel('Value')
        ax.set_ylabel('Frequency')
        ax.legend(loc='upper right', fontsize=8)
    
    plt.tight_layout()
    plt.show()

# Create interactive dropdown for histograms
hist_dropdown = widgets.Dropdown(
    options=selected_fields,
    description='Field:',
    style={'description_width': 'initial'},
    value=selected_fields[0]
)

hist_output = widgets.Output()
_init_mode_hist = True  # Prevent callback during init

def on_hist_field_change(change):
    with hist_output:
        clear_output(wait=True)
        plot_field_histograms(change['new'])

hist_dropdown.observe(on_hist_field_change, names='value')

print('Select a field from the dropdown to view histograms.')
display(widgets.VBox([
    widgets.HTML('<h3>Field Histogram Viewer</h3>'),
    hist_dropdown,
    hist_output
]))

print("Select a field from the dropdown to view histograms.")

Select a field from the dropdown to view histograms.


Select a field from the dropdown to view histograms.


## Section 6: Interactive Correlation Explorer

Explore pixel-by-pixel correlations between different data types. Select:
- **Field**: Which field to analyze
- **X-axis**: Dataset for the x-axis (e.g., satellite metric, terrain property)
- **Y-axis**: Dataset for the y-axis

The scatter plot shows correlation coefficient (r) and p-value (p).

In [6]:
# Prepare dataset options for correlation explorer
sat_options = {f'sat_{m}': ('satellite', m) for m in SATELLITE_METRICS}
terr_options = {f'terr_{p}': ('terrain', p) for p in TERRAIN_PROPERTIES}
soil_options = {f'soil_{p}': ('soil', p) for p in SOIL_PROPERTIES}

all_correlation_options = {**sat_options, **terr_options, **soil_options}
display_options = {
    **SATELLITE_DISPLAY,
    **TERRAIN_DISPLAY, 
    **SOIL_PROPERTY_DISPLAY
}

def get_data_label(key):
    """Get display label for a correlation key."""
    if key.startswith('sat_'):
        m = key.replace('sat_', '')
        return f'Satellite: {SATELLITE_DISPLAY.get(m, m.upper())}'
    elif key.startswith('terr_'):
        p = key.replace('terr_', '')
        return f'Terrain: {TERRAIN_DISPLAY.get(p, p.capitalize())}'
    elif key.startswith('soil_'):
        p = key.replace('soil_', '')
        return f'Soil: {SOIL_PROPERTY_DISPLAY.get(p, p)}'
    return key

def plot_correlation(field_id, x_key, y_key):
    """Plot pixel-by-pixel correlation between two datasets."""
    x_type, x_prop = all_correlation_options[x_key]
    y_type, y_prop = all_correlation_options[y_key]
    
    # Load data
    x_data, x_src = get_10m_data(field_id, x_type, x_prop)
    y_data, y_src = get_10m_data(field_id, y_type, y_prop)
    
    if x_data is None or y_data is None:
        print(f'Data not available for field {field_id}')
        return
    
    # Match shapes (take intersection)
    min_h = min(x_data.shape[0], y_data.shape[0])
    min_w = min(x_data.shape[1], y_data.shape[1])
    
    x_crop = x_data[:min_h, :min_w].flatten()
    y_crop = y_data[:min_h, :min_w].flatten()
    
    # Calculate correlation
    r, p, n = calculate_pixel_correlation(x_crop, y_crop)
    
    # Create scatter plot
    fig, ax = plt.subplots(figsize=(8, 6))
    
    valid_mask = ~np.isnan(x_crop) & ~np.isnan(y_crop)
    x_valid = x_crop[valid_mask]
    y_valid = y_crop[valid_mask]
    
    ax.scatter(x_valid, y_valid, alpha=0.3, s=10, c='steelblue', edgecolors='none')
    
    # Add regression line if valid correlation
    if not np.isnan(r) and np.std(x_valid) > 0 and np.std(y_valid) > 0:
        z = np.polyfit(x_valid, y_valid, 1)
        p_line = np.poly1d(z)
        x_line = np.linspace(x_valid.min(), x_valid.max(), 100)
        ax.plot(x_line, p_line(x_line), 'r-', lw=2, label=f'r = {r:.3f}')
        
        # Significance stars
        sig_star = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ''))
        ax.text(0.05, 0.95, f'p = {p:.2e} {sig_star}\nn = {n:,}', 
                transform=ax.transAxes, fontsize=11, va='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    ax.set_xlabel(get_data_label(x_key), fontsize=12)
    ax.set_ylabel(get_data_label(y_key), fontsize=12)
    ax.set_title(f'{field_id}: Correlation Analysis', fontsize=13, fontweight='bold')
    ax.legend(loc='lower right', fontsize=11)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Create dropdowns
corr_field_dropdown = widgets.Dropdown(
    options=selected_fields,
    description='Field:',
    style={'description_width': 'initial'},
    value=selected_fields[0]
)

x_axis_dropdown = widgets.Dropdown(
    options=list(all_correlation_options.keys()),
    description='X-axis:',
    style={'description_width': 'initial'},
    value='sat_ndvi'
)

y_axis_dropdown = widgets.Dropdown(
    options=list(all_correlation_options.keys()),
    description='Y-axis:',
    style={'description_width': 'initial'},
    value='terr_elevation'
)

corr_output = widgets.Output()
_init_mode_corr = True  # Prevent callback during init

def update_correlation(change):
    with corr_output:
        clear_output(wait=True)
        plot_correlation(
            corr_field_dropdown.value,
            x_axis_dropdown.value,
            y_axis_dropdown.value
        )

corr_field_dropdown.observe(update_correlation, names='value')
x_axis_dropdown.observe(update_correlation, names='value')
y_axis_dropdown.observe(update_correlation, names='value')

print('Select field and datasets to explore correlations.')
display(widgets.VBox([
    widgets.HTML('<h3>Pixel Correlation Explorer</h3>'),
    corr_field_dropdown,
    x_axis_dropdown,
    y_axis_dropdown,
    corr_output
]))

print("Select field and datasets to explore correlations.")

Select field and datasets to explore correlations.


Select field and datasets to explore correlations.


## Section 7: Correlation Matrix

Full correlation matrix showing pixel-by-pixel correlations between all data types (satellite, terrain, soil).
Use the dropdown to select which field to analyze.

In [ ]:
def plot_correlation_matrix(field_id):
    """Plot correlation matrix for all data types."""
    
    # Load all datasets
    datasets = {}
    
    # Satellite
    for m in SATELLITE_METRICS:
        data, src = get_10m_data(field_id, 'satellite', m)
        if data is not None:
            datasets[f'sat_{m}'] = data
    
    # Terrain
    for p in TERRAIN_PROPERTIES:
        data, src = get_10m_data(field_id, 'terrain', p)
        if data is not None:
            datasets[f'terr_{p}'] = data
    
    # Soil
    for p in SOIL_PROPERTIES:
        data, src = get_10m_data(field_id, 'soil', p)
        if data is not None:
            datasets[f'soil_{p}'] = data
    
    if len(datasets) < 2:
        print(f'Insufficient data for {field_id}')
        return
    
    # Find common shape
    min_h = min(d.shape[0] for d in datasets.values())
    min_w = min(d.shape[1] for d in datasets.values())
    
    # Compute correlation matrix
    dataset_names = list(datasets.keys())
    n_datasets = len(dataset_names)
    corr_matrix = np.full((n_datasets, n_datasets), np.nan)
    p_matrix = np.full((n_datasets, n_datasets), np.nan)
    
    for i, name_i in enumerate(dataset_names):
        for j, name_j in enumerate(dataset_names):
            if i == j:
                corr_matrix[i, j] = 1.0
                p_matrix[i, j] = 0.0
            elif i < j:
                data_i = datasets[name_i][:min_h, :min_w].flatten()
                data_j = datasets[name_j][:min_h, :min_w].flatten()
                r, p, n = calculate_pixel_correlation(data_i, data_j)
                corr_matrix[i, j] = r
                corr_matrix[j, i] = r
                p_matrix[i, j] = p
                p_matrix[j, i] = p
    
    # Create labels
    labels = []
    for name in dataset_names:
        if name.startswith('sat_'):
            m = name.replace('sat_', '')
            labels.append(f'Sat: {SATELLITE_DISPLAY.get(m, m.upper())}')
        elif name.startswith('terr_'):
            p = name.replace('terr_', '')
            labels.append(f'Terr: {TERRAIN_DISPLAY.get(p, p.capitalize())}')
        elif name.startswith('soil_'):
            p = name.replace('soil_', '')
            labels.append(f'Soil: {SOIL_PROPERTY_DISPLAY.get(p, p)}')
    
    # Plot heatmap
    fig, ax = plt.subplots(figsize=(14, 10))
    
    mask = np.isnan(corr_matrix)
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
                vmin=-1, vmax=1, xticklabels=labels, yticklabels=labels,
                mask=mask, ax=ax, annot_kws={'fontsize': 9})
    
    ax.set_title(f'{field_id} - Correlation Matrix', fontsize=14, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()
    
    # Print significant correlations
    print(f'\n=== Significant Correlations for {field_id} (p < 0.05) ===')
    sig_pairs = []
    for i in range(n_datasets):
        for j in range(i+1, n_datasets):
            if not np.isnan(p_matrix[i, j]) and p_matrix[i, j] < 0.05:
                sig_pairs.append({
                    'var1': labels[i],
                    'var2': labels[j],
                    'r': corr_matrix[i, j],
                    'p': p_matrix[i, j]
                })
    
    if sig_pairs:
        sig_df = pd.DataFrame(sig_pairs).sort_values('r', key=abs, ascending=False)
        print(sig_df.to_string(index=False))
    else:
        print('No significant correlations found.')

# Create dropdown
matrix_field_dropdown = widgets.Dropdown(
    options=selected_fields,
    description='Field:',
    style={'description_width': 'initial'},
    value=selected_fields[0]
)

matrix_output = widgets.Output()
_init_mode_matrix = True  # Prevent callback during init

def on_matrix_field_change(change):
    with matrix_output:
        clear_output(wait=True)
        plot_correlation_matrix(change['new'])

matrix_field_dropdown.observe(on_matrix_field_change, names='value')

print('Select a field to view its correlation matrix.')
display(widgets.VBox([
    widgets.HTML('<h3>Correlation Matrix Viewer</h3>'),
    matrix_field_dropdown,
    matrix_output
]))

print("Select a field to view its correlation matrix.")

Select a field to view its correlation matrix.


Select a field to view its correlation matrix.


## Section 8: Analysis & Summary

Summary of significant correlations and interpretation of results.

In [8]:

# Compute and summarize all correlations across all selected fields

all_results = []

print('Computing correlations for all fields...')

for field_id in selected_fields:
    print(f'  Processing {field_id}...')
    
    # Load all datasets
    datasets = {}
    
    for m in SATELLITE_METRICS:
        data, src = get_10m_data(field_id, 'satellite', m)
        if data is not None:
            datasets[f'sat_{m}'] = data
    
    for p in TERRAIN_PROPERTIES:
        data, src = get_10m_data(field_id, 'terrain', p)
        if data is not None:
            datasets[f'terr_{p}'] = data
    
    for p in SOIL_PROPERTIES:
        data, src = get_10m_data(field_id, 'soil', p)
        if data is not None:
            datasets[f'soil_{p}'] = data
    
    if len(datasets) < 2:
        continue
    
    # Find common shape
    min_h = min(d.shape[0] for d in datasets.values())
    min_w = min(d.shape[1] for d in datasets.values())
    
    dataset_names = list(datasets.keys())
    
    # Compute correlations
    for i, name_i in enumerate(dataset_names):
        for j, name_j in enumerate(dataset_names):
            if i < j:
                data_i = datasets[name_i][:min_h, :min_w].flatten()
                data_j = datasets[name_j][:min_h, :min_w].flatten()
                r, p, n = calculate_pixel_correlation(data_i, data_j)
                
                # Categorize
                var1_type = 'Satellite' if name_i.startswith('sat_') else ('Terrain' if name_i.startswith('terr_') else 'Soil')
                var2_type = 'Satellite' if name_j.startswith('sat_') else ('Terrain' if name_j.startswith('terr_') else 'Soil')
                
                all_results.append({
                    'field_id': field_id,
                    'var1': name_i,
                    'var2': name_j,
                    'var1_type': var1_type,
                    'var2_type': var2_type,
                    'r': r,
                    'p': p,
                    'n': n,
                    'significant': 'Yes' if p < 0.05 else 'No'
                })

results_df = pd.DataFrame(all_results)
print(f'\nTotal correlations computed: {len(results_df)}')

# Filter significant correlations
sig_results = results_df[results_df['significant'] == 'Yes'].copy()
sig_results = sig_results.sort_values('p')

print(f'Significant correlations (p < 0.05): {len(sig_results)}')

# Display summary by category
print('\n=== Summary by Category ===')

for cat1, cat2 in [('Satellite', 'Terrain'), ('Satellite', 'Soil'), ('Terrain', 'Soil')]:
    subset = sig_results[(sig_results['var1_type'] == cat1) & (sig_results['var2_type'] == cat2)]
    print(f'\n{cat1} vs {cat2}: {len(subset)} significant correlations')
    if len(subset) > 0:
        top_corr = subset.nlargest(3, 'r', keep='first')
        for _, row in top_corr.iterrows():
            print(f"  {row['field_id']}: r={row['r']:.3f}, p={row['p']:.2e}")


Computing correlations for all fields...
  Processing WV_AG_041...
  Processing WV_AG_008...
  Processing WV_AG_002...
  Processing WV_AG_048...

Total correlations computed: 364
Significant correlations (p < 0.05): 153

=== Summary by Category ===

Satellite vs Terrain: 34 significant correlations
  WV_AG_002: r=0.298, p=0.00e+00
  WV_AG_002: r=0.241, p=0.00e+00
  WV_AG_002: r=0.239, p=0.00e+00

Satellite vs Soil: 18 significant correlations
  WV_AG_002: r=0.198, p=0.00e+00
  WV_AG_002: r=0.198, p=0.00e+00
  WV_AG_002: r=0.090, p=1.97e-14

Terrain vs Soil: 36 significant correlations
  WV_AG_002: r=0.437, p=0.00e+00
  WV_AG_002: r=0.437, p=0.00e+00
  WV_AG_002: r=0.437, p=0.00e+00


/tmp/ipykernel_574784/230212368.py:55: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = stats.pearsonr(x, y)


In [9]:
# Display table of significant correlations
print('=== Table of Significant Correlations (p < 0.05) ===\n')

# Format for display
display_df = sig_results[['field_id', 'var1', 'var2', 'r', 'p', 'n']].copy()
display_df['r'] = display_df['r'].round(3)
display_df['p'] = display_df['p'].apply(lambda x: f'{x:.2e}')
display_df['n'] = display_df['n'].apply(lambda x: f'{x:,}')
display_df.columns = ['Field', 'Variable 1', 'Variable 2', 'r', 'p-value', 'n pixels']

display(display_df.head(20))

if len(sig_results) > 20:
    print(f'\n... showing first 20 of {len(sig_results)} significant correlations')


=== Table of Significant Correlations (p < 0.05) ===



,Field,Variable 1,Variable 2,r,p-value,n pixels
0,WV_AG_041,sat_ndvi,sat_msavi,0.784,0.00e+00,"1,410"
1,WV_AG_041,sat_ndvi,sat_evi,0.766,0.00e+00,"1,410"
2,WV_AG_041,sat_ndvi,sat_ndmi,0.867,0.00e+00,"1,410"
13,WV_AG_041,sat_msavi,sat_evi,0.999,0.00e+00,"1,410"
15,WV_AG_041,sat_msavi,terr_elevation,-0.624,0.00e+00,"1,410"
14,WV_AG_041,sat_msavi,sat_ndmi,0.964,0.00e+00,"1,410"
26,WV_AG_041,sat_evi,terr_elevation,-0.631,0.00e+00,"1,410"
25,WV_AG_041,sat_evi,sat_ndmi,0.960,0.00e+00,"1,410"
36,WV_AG_041,sat_ndmi,terr_elevation,-0.534,0.00e+00,"1,410"
137,WV_AG_008,terr_elevation,terr_slope,-0.843,0.00e+00,"1,479"



... showing first 20 of 153 significant correlations


## Summary & Interpretation

### Key Findings

This analysis examined pixel-by-pixel correlations between satellite vegetation indices (NDVI, MSAVI, EVI, NDMI), terrain properties (elevation, slope, aspect), and soil properties (pH, organic matter, clay, sand, CEC, AWC, silt) across 4 randomly selected agricultural fields in Oregon's Willamette Valley.

#### Satellite-Terrain Relationships
- **Elevation**: Vegetation indices show varying relationships with elevation across fields
- **Slope**: Steeper areas may have different vegetation patterns due to drainage, soil depth, or management
- **Aspect**: North-facing vs south-facing slopes receive different solar radiation, affecting plant growth

#### Satellite-Soil Relationships
- Soil properties influence vegetation as they affect water retention, nutrient availability, and root development
- Correlations with soil are typically weaker than terrain due to soil data resolution (5m vs finer satellite resolution)

#### Notable Patterns
- Fields showing strong positive correlations between NDVI and elevation suggest higher ground vegetation is healthier
- Terrain properties generally show stronger correlations than soil properties

### Limitations

1. **Sample size**: Only 4 fields analyzed
2. **Temporal mismatch**: Satellite imagery from specific dates may not capture full growing season
3. **Soil data resolution**: SSURGO soil data is coarse (5m) compared to satellite (10m)

### Conclusions

The analysis demonstrates that terrain properties (especially elevation and slope) have detectable relationships with vegetation indices. This suggests that remote sensing can effectively capture topographical influences on crop health. Further analysis with more fields and temporal data would strengthen these findings.

---
*Analysis completed using Sentinel-2 satellite imagery, SSURGO soil data, and USGS terrain data.*
